In [3]:
# eval_retrieval.py
import torch
import numpy as np
from PIL import Image
from pathlib import Path
import json
from model import longclip

# ── Config ───────────────────────────────────────────────────────────────────
CHECKPOINT   = "./checkpoints/DGTRS-CLIP-ViT-B-16.pt"
DATASET_JSON = "./data/dataset_RSITMD.json"
IMAGE_DIR    = "./images/rsitmd/"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
KS           = [1, 5, 10]

# ── Load model ───────────────────────────────────────────────────────────────
model, preprocess = longclip.load(CHECKPOINT, device=DEVICE)
model.eval()

# ── Load dataset — YOUR exact structure ──────────────────────────────────────
with open(DATASET_JSON) as f:
    raw_data = json.load(f)

# Top-level key is "images", filter test split only
data = [item for item in raw_data["images"] if item["split"] == "test"]

print(f"Test split: {len(data)} images")

images_raw, all_captions, img_caption_map = [], [], {}

for idx, item in enumerate(data):
    images_raw.append(item["filename"])          # e.g. "baseballfield_452.tif"
    for sent in item["sentences"]:               # key: "sentences"
        img_caption_map[len(all_captions)] = idx
        all_captions.append(sent["raw"])         # key: "raw"

print(f"Total captions: {len(all_captions)}")    # should be ~5x images

# ── Encode all images ─────────────────────────────────────────────────────────
print(f"\nEncoding {len(images_raw)} images...")
image_features = []
with torch.no_grad():
    for fname in images_raw:
        img_path = Path(IMAGE_DIR) / fname       # handles .tif correctly
        img = preprocess(Image.open(img_path)).unsqueeze(0).to(DEVICE)
        feat = model.encode_image(img)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        image_features.append(feat.cpu())
image_features = torch.cat(image_features, dim=0)

# ── Encode all captions ───────────────────────────────────────────────────────
print(f"Encoding {len(all_captions)} captions...")
BATCH = 128
text_features = []
with torch.no_grad():
    for i in range(0, len(all_captions), BATCH):
        batch = all_captions[i:i+BATCH]
        tokens = longclip.tokenize(batch).to(DEVICE)
        feat = model.encode_text(tokens)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        text_features.append(feat.cpu())
text_features = torch.cat(text_features, dim=0)

# ── Similarity matrix ─────────────────────────────────────────────────────────
sim_matrix = image_features @ text_features.T    # (N_images, N_captions)

# ── Recall@K helper ───────────────────────────────────────────────────────────
def recall_at_k(ranked_indices, ground_truth_set, k):
    return int(len(set(ranked_indices[:k]) & ground_truth_set) > 0)

# ── Image → Text ──────────────────────────────────────────────────────────────
print("\nComputing Image→Text Recall@K...")
i2t_scores = {k: [] for k in KS}

for img_idx in range(len(images_raw)):
    gt_caps = {c_idx for c_idx, i_idx in img_caption_map.items() if i_idx == img_idx}
    ranked = sim_matrix[img_idx].argsort(descending=True).tolist()
    for k in KS:
        i2t_scores[k].append(recall_at_k(ranked, gt_caps, k))

# ── Text → Image ──────────────────────────────────────────────────────────────
print("Computing Text→Image Recall@K...")
t2i_scores = {k: [] for k in KS}

for cap_idx in range(len(all_captions)):
    gt_img = {img_caption_map[cap_idx]}
    ranked = sim_matrix[:, cap_idx].argsort(descending=True).tolist()
    for k in KS:
        t2i_scores[k].append(recall_at_k(ranked, gt_img, k))

# ── Print results ─────────────────────────────────────────────────────────────
print("\n" + "="*40)
print(f"{'Metric':<18} {'Score':>8}")
print("="*40)
for k in KS:
    print(f"  I→T  R@{k:<3}         {np.mean(i2t_scores[k])*100:>6.2f}%")
for k in KS:
    print(f"  T→I  R@{k:<3}         {np.mean(t2i_scores[k])*100:>6.2f}%")

mean_r = np.mean(
    [np.mean(i2t_scores[k]) for k in KS] +
    [np.mean(t2i_scores[k]) for k in KS]
) * 100
print("="*40)
print(f"  Mean Recall      {mean_r:>6.2f}%")
print("="*40)

Test split: 452 images
Total captions: 2260

Encoding 452 images...
Encoding 2260 captions...

Computing Image→Text Recall@K...
Computing Text→Image Recall@K...

Metric                Score
  I→T  R@1            21.02%
  I→T  R@5            38.94%
  I→T  R@10           51.99%
  T→I  R@1            15.22%
  T→I  R@5            40.97%
  T→I  R@10           55.22%
  Mean Recall       37.23%
